# 02d Estimate: the regression models plus the extra data blocks

**Purpose:** Add each extra data block of `01c_preprocess_extra_data.ipynb` to the three regression models of `02b_estimate_regression.ipynb`, one block at a time, and then all blocks together, in the same backtest. Everything else stays as in `02b` and `02c`, so that any gain or loss can be attributed to the block.
**Input:** `../data/processed/asylum_clean.csv`, `../data/processed/extra_indicators_clean.csv`, `../data/processed/nationality_shares_clean.csv`, `../data/processed/conflict_intensity_clean.csv` (from `01c_preprocess_extra_data.ipynb`).
**Output:** `../data/processed/backtest_forecasts_extra.csv` (same columns as `backtest_forecasts_indicator.csv`, plus `block`).

The test period (2024-10 to 2026-09) is **not used**: it was already used once in `04_test.ipynb`. We compare the models in `03_evaluate.ipynb` (Part B).

In [1]:
import numpy as np
import pandas as pd
import warnings

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

np.random.seed(42)

## 1. Load the target and the extra data

As in `02c`, `reindex` puts every series on the months of the target.

In [2]:
clean = pd.read_csv("../data/processed/asylum_clean.csv")
clean["month"] = pd.to_datetime(clean["month"])
clean = clean.set_index("month")
months = clean.index
y = clean["applicants_total"].to_numpy(dtype=float)
log_y = clean["log_applicants"].to_numpy(dtype=float)

def load(path):
    data = pd.read_csv(path)
    data["month"] = pd.to_datetime(data["month"])
    return data.set_index("month")

extra = load("../data/processed/extra_indicators_clean.csv").reindex(months)
shares = load("../data/processed/nationality_shares_clean.csv").reindex(months)
intensity = load("../data/processed/conflict_intensity_clean.csv")

log_neighbours = extra["log_neighbours"].to_numpy(dtype=float)
log_med = extra["log_med_routes"].to_numpy(dtype=float)

FIRST_ORIGIN = pd.Timestamp("2012-12-01")
LAST_BACKTEST = pd.Timestamp("2024-09-01")
H_MAX = 3
first_origin = int(np.where(months == FIRST_ORIGIN)[0][0])
last_backtest = int(np.where(months == LAST_BACKTEST)[0][0])
print("Neighbours missing:", np.isnan(log_neighbours).sum(), "| Med routes missing:", np.isnan(log_med).sum())

Neighbours missing: 1 | Med routes missing: 12


## 2. The conflict index

For a forecast origin $t$, with $w_{c,t}$ the share of nationality $c$ in the Swiss applications of the 12 months up to $t$ (top 10 nationalities only) and $I_{c,s} = \log(1 + \text{deaths in country } c \text{ in month } s)$:

$$\text{conflict}_k(t) = \sum_{c \in \text{top 10 at } t} w_{c,t} \, \big(I_{c,t-1} - I_{c,t-1-k}\big)$$

It measures **how much the conflicts got worse** over the last $k$ known months, in the countries people came from recently. The weights stay fixed inside the formula, so a change in the Swiss nationality mix does not move the index. We decided on this formula and on $k = 3$ and $k = 6$ before running any model.

In [3]:
def conflict_change(t, k):
    w = shares.iloc[t]
    if w.isna().all():
        return np.nan
    top = w.nlargest(10)
    m1 = months[t] - pd.DateOffset(months=1)
    m0 = months[t] - pd.DateOffset(months=1 + k)
    if m1 not in intensity.index or m0 not in intensity.index:
        return np.nan
    change = intensity.loc[m1, top.index] - intensity.loc[m0, top.index]
    return float((top * change).sum())

conflict_3 = np.array([conflict_change(t, 3) for t in range(len(months))])
conflict_6 = np.array([conflict_change(t, 6) for t in range(len(months))])
print("Conflict index available from", months[np.argmax(~np.isnan(conflict_3))].strftime("%Y-%m"))
pd.DataFrame({"conflict_3_months": conflict_3, "conflict_6_months": conflict_6}, index=months).loc[
    ["2015-06-01", "2021-08-01", "2021-09-01", "2021-10-01", "2023-11-01"]].round(3)

Conflict index available from 2008-12


,conflict_3_months,conflict_6_months
month,,
2015-06-01,0.001,0.036
2021-08-01,0.080,0.002
2021-09-01,-0.006,0.676
2021-10-01,-0.677,-0.285
2023-11-01,0.449,0.211


**What we see.** The index exists from December 2008 (the first origin with 12 months of nationality data). It reacts to known events: after the fall of Kabul, the 6-month escalation jumps to 0.68 at the origin of September 2021 (August deaths against February), when Afghans were 19% of the Swiss applications. In October 2021 the 3-month index falls to −0.68, because the fighting stopped. In normal months the index is close to 0.

## 3. The blocks

Each block adds two features, like the route block of `02c`: the newest change and the change over three months (for the conflict index: over 3 and 6 months), ending at the newest month known at the cutoff (`01c`, section 6):

| Block | Features | Newest month used |
|---|---|---|
| neighbours | $\log N_{t-1} - \log N_{t-2}$, $\log N_{t-1} - \log N_{t-4}$ ($N$ = applications DE + AT + FR + IT) | $t-1$ |
| Med routes | $\log M_t - \log M_{t-1}$, $\log M_t - \log M_{t-3}$ ($M$ = 1 + detections on the four Mediterranean routes) | $t$ (UNHCR timing) |
| conflict | $\text{conflict}_3(t)$, $\text{conflict}_6(t)$ | $t-1$ |
| all three | the six features above | |

We fixed these blocks before running any model. As in `02c`, the other features, the target, the models, the grids of `alpha` and the validation inside each training window are **exactly those of `02b`**.

In [4]:
blocks = {
    "neighbours": {
        "neighbours_change_last_month": lambda t: log_neighbours[t - 1] - log_neighbours[t - 2],
        "neighbours_change_3_months": lambda t: log_neighbours[t - 1] - log_neighbours[t - 4],
    },
    "Med routes": {
        "med_change_last_month": lambda t: log_med[t] - log_med[t - 1],
        "med_change_3_months": lambda t: log_med[t] - log_med[t - 3],
    },
    "conflict": {
        "conflict_3_months": lambda t: conflict_3[t],
        "conflict_6_months": lambda t: conflict_6[t],
    },
}
blocks["all three"] = {**blocks["neighbours"], **blocks["Med routes"], **blocks["conflict"]}

def make_table(h, block):
    # Same table as in 02b, plus the features of one block (never later than the month known at the cutoff)
    extra_features = blocks[block]
    rows = []
    for t in range(12, len(log_y) - h):
        values = {name: f(t) for name, f in extra_features.items()}
        if any(np.isnan(v) for v in values.values()):     # a block starts when all its months exist
            continue
        rows.append({
            "t": t,
            "change_last_month": log_y[t] - log_y[t - 1],
            "change_month_before": log_y[t - 1] - log_y[t - 2],
            "same_change_last_year": log_y[t + h - 12] - log_y[t - 12],
            **values,
            "calendar_month": months[t + h].month,
            "target": log_y[t + h] - log_y[t],
        })
    table = pd.DataFrame(rows).set_index("t")
    dummies = pd.get_dummies(table["calendar_month"], prefix="month", drop_first=True).astype(float)
    features = ["change_last_month", "change_month_before", "same_change_last_year"] + list(extra_features)
    X = pd.concat([table[features], dummies], axis=1)
    return X, table["target"]

tables = {(block, h): make_table(h, block) for block in blocks for h in range(1, H_MAX + 1)}
for block in blocks:
    X2, _ = tables[(block, 2)]
    print(f"{block:11s} h = 2: {X2.shape[0]} rows, {X2.shape[1]} features, first origin {months[X2.index.min()]:%Y-%m}, "
          f"missing values {X2.isna().sum().sum()}")

neighbours  h = 2: 211 rows, 16 features, first origin 2009-01, missing values 0
Med routes  h = 2: 208 rows, 16 features, first origin 2009-04, missing values 0
conflict    h = 2: 211 rows, 16 features, first origin 2009-01, missing values 0
all three   h = 2: 208 rows, 20 features, first origin 2009-04, missing values 0


**What we see.** For $h = 2$, each table has 16 features (20 with all three blocks) and no missing value. The neighbours and conflict tables start at the origin January 2009, like `02b`; the tables with the Mediterranean routes start in April 2009, because the routes start in January 2009 and the 3-month change needs month $t-3$ (3 fewer training rows than in `02b`).

## 4. Models, grids and the choice of `alpha`: the same as in `02b`

Copied from `02c` without any change.

In [5]:
def make_model(name, alpha=None):
    if name == "Linear regression":
        return make_pipeline(StandardScaler(), LinearRegression())
    if name == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    return make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=10000))

alpha_grids = {
    "Linear regression": [None],
    "Ridge": [0.1, 1, 10, 100, 1000, 10000],
    "Lasso": [0.0001, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3],
}

def choose_alpha(name, X, target, T, h, n_blocks=2, block=12):
    grid = alpha_grids[name]
    if len(grid) == 1:
        return grid[0]
    first_row, last_row = X.index.min(), T - h
    scores = []
    for alpha in grid:
        mse_blocks = []
        for b in range(n_blocks):
            start = last_row - n_blocks * block + 1 + b * block
            valid_rows = list(range(start, start + block))
            train_rows = list(range(first_row, start - h + 1))
            model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
            errors = target.loc[valid_rows].to_numpy() - model.predict(X.loc[valid_rows])
            mse_blocks.append(np.mean(errors ** 2))
        scores.append(np.mean(mse_blocks))
    return grid[int(np.argmin(scores))]

## 5. The backtest loop

The same loop as in `02b` and `02c`, once per block: the same 141 origins and horizons, and the same check that no training target lies after the origin (`assert`). This takes about four minutes.

In [6]:
rows = []
for block in blocks:
    for t in range(first_origin, last_backtest):
        for h in range(1, H_MAX + 1):
            target_pos = t + h
            if target_pos > last_backtest:
                continue
            X, target = tables[(block, h)]
            train_rows = list(range(X.index.min(), t - h + 1))
            assert max(train_rows) + h <= t                    # no training target after the origin

            for name in alpha_grids:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    alpha = choose_alpha(name, X, target, t, h)
                    model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
                forecast_log = log_y[t] + model.predict(X.loc[[t]])[0]
                rows.append({
                    "model": f"{name} + {block}", "block": block, "origin": months[t], "month": months[target_pos],
                    "h": h, "forecast_log": forecast_log, "actual_log": log_y[target_pos],
                    "forecast": np.exp(forecast_log), "actual": y[target_pos], "alpha": alpha,
                })

forecasts = pd.DataFrame(rows)
print("Number of stored forecasts:", forecasts.shape[0])

Number of stored forecasts: 5040


## 6. Checks on the stored forecasts

In [7]:
print(forecasts.groupby(["block", "h"]).size().unstack())
print()
print("All forecast months in the backtest (before 2024-10):", (forecasts["month"] < pd.Timestamp("2024-10-01")).all())
print("All forecast months after their origin:", (forecasts["month"] > forecasts["origin"]).all())
print("Missing or infinite forecasts:", (~np.isfinite(forecasts["forecast_log"])).sum())

h             1    2    3
block                    
Med routes  423  420  417
all three   423  420  417
conflict    423  420  417
neighbours  423  420  417

All forecast months in the backtest (before 2024-10): True
All forecast months after their origin: True
Missing or infinite forecasts: 0


**What we see.** As in `02b` and `02c`: for each block, 141, 140 and 139 forecasts per model for $h = 1, 2, 3$ (423, 420 and 417 for the three models), 5,040 together, all in the backtest, all after their origin, none missing. With the `assert` in the loop, this shows that the extra blocks used only data known at each origin and never touched the test period.

### Which penalties were chosen?

As in `02c` (notebook 209), we check that the chosen `alpha` does not sit mostly at an edge of the grid.

In [8]:
for name in ["Ridge", "Lasso"]:
    chosen = forecasts[forecasts["model"].str.startswith(name + " +")]
    print(f"{name}: how often each alpha was chosen, per block")
    print(chosen.groupby(["block", "alpha"]).size().unstack(fill_value=0).to_string())
    print()

Ridge: how often each alpha was chosen, per block
alpha       0.1      1.0      10.0     100.0    1000.0   10000.0
block                                                           
Med routes       55        9       66      269       20        1
all three        24        5       85      271       33        2
conflict         85        6       45      217       61        6
neighbours       63       25       72      174       56       30

Lasso: how often each alpha was chosen, per block
alpha       0.0001  0.0010  0.0030  0.0100  0.0300  0.1000  0.3000
block                                                             
Med routes      57      33      16     165      70      30      49
all three       49      16      48     127      96      34      50
conflict        83      11      23     100     115      59      29
neighbours      55      28      43      92      99      74      29



**What we see.** Ridge most often chose `alpha = 100` in every block (174 to 271 of 420 times); the largest value (10,000) was chosen at most 30 times (neighbours). Lasso most often chose 0.01 or 0.03; its largest value (0.3) was chosen 29 to 50 times.

**What this means for us.** As in `02c`, the upper edges are rarely reached, and the lower edge is almost the linear regression, which we run anyway. We keep the grids of `02b`, so that only the data block changes.

## 7. What do the models do with the extra data?

As in `02c`, the coefficients for $h = 2$ at the last origin (August 2024), on standardised features (notebook 105), for the block with all three sources. Lasso can set a coefficient to exactly 0 (notebook 105), so it shows which sources it keeps. A coefficient says what helps to *predict*, not what *causes*.

In [9]:
t = last_backtest - 1
h = 2
X, target = tables[("all three", h)]
train_rows = list(range(X.index.min(), t - h + 1))

coefficients = pd.DataFrame(index=X.columns)
for name in ["Linear regression", "Ridge", "Lasso"]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        alpha = choose_alpha(name, X, target, t, h)
        model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
    coefficients[f"{name} (alpha = {alpha})"] = model[-1].coef_

print("Origin:", months[t], "| horizon h =", h, "| training rows:", len(train_rows))
coefficients.iloc[:9].round(3)

Origin: 2024-08-01 00:00:00 | horizon h = 2 | training rows: 183


,Linear regression (alpha = None),Ridge (alpha = 10),Lasso (alpha = 0.003)
change_last_month,-0.039,-0.035,-0.031
change_month_before,-0.018,-0.015,-0.008
same_change_last_year,0.017,0.019,0.018
neighbours_change_last_month,0.014,0.007,0.000
neighbours_change_3_months,-0.062,-0.055,-0.052
med_change_last_month,0.051,0.047,0.047
med_change_3_months,0.035,0.035,0.034
conflict_3_months,0.022,0.019,0.014
conflict_6_months,-0.026,-0.022,-0.015


Lasso at every 12th origin: which extra features does it keep (coefficient not 0)?

In [10]:
extra_names = list(blocks["all three"])
kept = []
for t in range(first_origin, last_backtest - 2, 12):
    X, target = tables[("all three", 2)]
    train_rows = list(range(X.index.min(), t - 2 + 1))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        alpha = choose_alpha("Lasso", X, target, t, 2)
        model = make_model("Lasso", alpha).fit(X.loc[train_rows], target.loc[train_rows])
    coef = pd.Series(model[-1].coef_, index=X.columns)
    kept.append({"origin": months[t].strftime("%Y-%m"), "alpha": alpha, **coef[extra_names].round(3).to_dict()})
pd.DataFrame(kept).set_index("origin")

,alpha,neighbours_change_last_month,neighbours_change_3_months,med_change_last_month,med_change_3_months,conflict_3_months,conflict_6_months
origin,,,,,,,
2012-12,0.0300,0.000,-0.000,0.000,0.020,-0.000,0.000
2013-12,0.0300,0.000,-0.000,0.000,0.006,0.000,0.000
2014-12,0.0100,0.000,-0.024,-0.001,0.049,-0.000,0.000
2015-12,0.0300,0.000,-0.000,0.000,0.074,-0.000,0.000
2016-12,0.0300,0.000,-0.000,0.014,0.054,0.000,0.000
2017-12,0.0030,0.020,-0.018,0.040,0.050,0.000,-0.012
2018-12,0.0030,0.026,-0.018,0.029,0.055,-0.000,-0.007
2019-12,0.1000,0.000,0.000,0.000,0.000,0.000,0.000
2020-12,0.0001,0.008,-0.080,0.057,0.043,0.030,-0.017


**What we see.**
- At the last origin (183 training rows), the three models agree. The largest extra coefficients are the two **Mediterranean route** changes (+0.047 and +0.034 for Lasso) and the 3-month change of the **neighbours** (−0.052). The conflict features are smaller (+0.014 and −0.015).
- Over the years, Lasso keeps the 3-month route change at every origin shown except 2019, and the monthly route change at most origins from 2016. It sets both conflict features to 0 in 8 of the 12 origins shown, and mostly drops the neighbours until 2017.
- The two neighbour features have opposite signs, as the two route features had in `02c`, so we should not read each sign on its own.

**What this means for us.** If one extra source carries information, it is the Mediterranean routes at the UNHCR timing. Whether the blocks improve the *forecasts* is checked in `03_evaluate.ipynb` (Part B).

## 8. Save the forecasts

In [11]:
forecasts.to_csv("../data/processed/backtest_forecasts_extra.csv", index=False)
print("Saved", forecasts.shape[0], "forecasts with the columns", list(forecasts.columns))

Saved 5040 forecasts with the columns ['model', 'block', 'origin', 'month', 'h', 'forecast_log', 'actual_log', 'forecast', 'actual', 'alpha']


## Summary

- **Models:** the three regressions of `02b`, with the same features, target, grids and validation, plus one block of extra features at a time: the neighbours (Eurostat, up to $t-1$), the Mediterranean routes (Frontex numbers at the UNHCR timing $t$), the conflict index (UCDP deaths weighted by the nationalities of the last 12 months, up to $t-1$), and all three blocks together.
- **Conflict index:** the weighted change of log deaths over 3 and 6 months, with the weights fixed at the origin; it reacts to the fall of Kabul (0.68).
- **Rules as before:** the same 141 origins and horizons; scaling and the choice of `alpha` inside each training window; no training target after the origin (`assert`); the test period is not used.
- **Lasso keeps mainly the Mediterranean routes**, and mostly sets the conflict index to 0.
- **Output:** 5,040 forecasts in `../data/processed/backtest_forecasts_extra.csv`, compared in `03_evaluate.ipynb` (Part B).